# <font color="#418FDE" size="6.5" uppercase>**Advanced Neighbors**</font>

>Last update: 20260712.
    
By the end of this Lecture, you will be able to:
- Use centroid and graph-based neighbor tools for classification and representation tasks. 
- Construct k-neighbor and radius-neighbor graphs for downstream algorithms. 
- Apply Neighborhood Components Analysis to learn transformations that improve neighbor classification. 


## **1. Centroid Classification Methods**

### **1.1. Nearest Centroid Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_B/image_01_01.jpg?v=1783845518" width="250">



>* Classes are summarized by average centroids.
>* New cases join the nearest profile.

>* Scale features; centroids work for coherent clusters.
>* Outliers or complex shapes weaken centroid accuracy.

>* Fast, simple baseline using class averages.
>* Centroids summarize groups for practical decisions.



In [ ]:
#@title Python Code - Nearest Centroid Basics

# This script demonstrates nearest centroid classification.
# We compute class centers manually.
# A tiny plot shows decision intuition.

import numpy as np
import matplotlib.pyplot as plt

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Create two small training groups.
class_a = rng.normal(loc=[2.0, 2.0], scale=0.35, size=(8, 2))
class_b = rng.normal(loc=[5.0, 4.5], scale=0.35, size=(8, 2))

# Validate expected training shapes.
assert class_a.shape == (8, 2)
assert class_b.shape == (8, 2)

# Compute one centroid per class.
centroid_a = class_a.mean(axis=0)
centroid_b = class_b.mean(axis=0)

# Define one new observation.
new_point = np.array([3.4, 3.1])

# Measure Euclidean distances safely.
dist_a = np.linalg.norm(new_point - centroid_a)
dist_b = np.linalg.norm(new_point - centroid_b)

# Choose the nearest centroid label.
predicted = "Class A" if dist_a < dist_b else "Class B"

# Print a compact teaching summary.
print("Centroid A:", np.round(centroid_a, 2))
print("Centroid B:", np.round(centroid_b, 2))
print("New point:", np.round(new_point, 2))
print("Distance to A:", round(float(dist_a), 2))
print("Distance to B:", round(float(dist_b), 2))
print("Predicted label:", predicted)

# Plot points, centroids, and new case.
plt.figure(figsize=(6, 4))
plt.scatter(class_a[:, 0], class_a[:, 1], label="Class A", color="steelblue")
plt.scatter(class_b[:, 0], class_b[:, 1], label="Class B", color="darkorange")
plt.scatter(*centroid_a, marker="X", s=180, color="navy")

plt.scatter(*centroid_b, marker="X", s=180, color="saddlebrown")
plt.scatter(*new_point, marker="*", s=220, color="crimson", label="New point")
plt.plot([new_point[0], centroid_a[0]], [new_point[1], centroid_a[1]], "--", color="navy")
plt.plot([new_point[0], centroid_b[0]], [new_point[1], centroid_b[1]], "--", color="saddlebrown")

# Add labels for beginner clarity.
plt.title("Nearest Centroid Basics")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()

plt.tight_layout()
plt.show()



### **1.2. Centroid Similarity**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_B/image_01_02.jpg?v=1783845520" width="250">



>* Centroids summarize each class's typical feature pattern.
>* Similarity depends on features and distance choice.

>* Feature scaling shapes centroid similarity meaning.
>* Well-prepared features make classes interpretable.

>* Centroids compactly represent class relationships.
>* Single centroids can oversimplify diverse classes.



In [ ]:
#@title Python Code - Centroid Similarity

# This script demonstrates centroid similarity clearly.
# We compare points with class average profiles.
# Scaling changes which centroid seems closest.

import numpy as np
import matplotlib.pyplot as plt

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Create two simple labeled groups.
class_a = np.array([[1.0, 20.0], [2.0, 22.0], [1.5, 19.0]])
class_b = np.array([[8.0, 80.0], [9.0, 78.0], [8.5, 82.0]])

# Compute one centroid per class.
centroid_a = class_a.mean(axis=0)
centroid_b = class_b.mean(axis=0)

# Define a new observation.
new_point = np.array([5.0, 40.0])

# Check expected feature dimensions.
assert class_a.shape[1] == 2 and class_b.shape[1] == 2
assert new_point.shape[0] == 2

# Measure Euclidean distance to centroids.
dist_a = np.linalg.norm(new_point - centroid_a)
dist_b = np.linalg.norm(new_point - centroid_b)

# Convert distance into simple similarity.
sim_a = 1.0 / (1.0 + dist_a)
sim_b = 1.0 / (1.0 + dist_b)

# Standardize using all observed points.
all_points = np.vstack([class_a, class_b, new_point])
means = all_points.mean(axis=0)
stds = all_points.std(axis=0)

# Guard against zero standard deviation.
stds = np.where(stds == 0, 1.0, stds)

# Scale classes, centroids, and new point.
class_a_scaled = (class_a - means) / stds
class_b_scaled = (class_b - means) / stds
new_scaled = (new_point - means) / stds

# Recompute centroids after scaling.
centroid_a_scaled = class_a_scaled.mean(axis=0)
centroid_b_scaled = class_b_scaled.mean(axis=0)

# Measure scaled distances and similarities.
dist_a_scaled = np.linalg.norm(new_scaled - centroid_a_scaled)
dist_b_scaled = np.linalg.norm(new_scaled - centroid_b_scaled)
sim_a_scaled = 1.0 / (1.0 + dist_a_scaled)
sim_b_scaled = 1.0 / (1.0 + dist_b_scaled)

# Choose the closer centroid each time.
raw_choice = "A" if dist_a < dist_b else "B"
scaled_choice = "A" if dist_a_scaled < dist_b_scaled else "B"

# Print a compact teaching summary.
print("Centroid A:", np.round(centroid_a, 2))
print("Centroid B:", np.round(centroid_b, 2))
print("New point:", new_point)
print("Raw similarities A/B:", round(sim_a, 3), round(sim_b, 3))
print("Closest centroid without scaling:", raw_choice)
print("Scaled similarities A/B:", round(sim_a_scaled, 3), round(sim_b_scaled, 3))
print("Closest centroid with scaling:", scaled_choice)

# Plot points, centroids, and new observation.
plt.figure(figsize=(6, 4))
plt.scatter(class_a[:, 0], class_a[:, 1], label="Class A", s=60)
plt.scatter(class_b[:, 0], class_b[:, 1], label="Class B", s=60)
plt.scatter(*centroid_a, marker="X", s=140, label="Centroid A")
plt.scatter(*centroid_b, marker="X", s=140, label="Centroid B")

# Highlight the new point.
plt.scatter(*new_point, color="black", marker="*", s=180, label="New point")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Centroid similarity depends on representation")
plt.legend()

# Show the single allowed plot.
plt.tight_layout()
plt.show()



### **1.3. Centroid Classifiers**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_B/image_01_03.jpg?v=1783845522" width="250">



>* Classify by nearest class prototype.
>* Fast, interpretable across medicine and documents.

>* Classify and summarize using class centers.
>* Centroids compress data and reduce noise.

>* Centroids work best for single, tight clusters.
>* Scaling, distance, and context shape usefulness.



## **2. Neighbor Graphs**

### **2.1. K Neighbor Graphs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_B/image_02_01.jpg?v=1783845514" width="250">



>* Connects each point to k nearest.
>* k controls locality for graph-based methods.

>* Equal neighbors give balanced local comparisons.
>* Nearest links reduce complexity across applications.

>* Choosing k balances fragmentation and overconnection.
>* Good graphs preserve meaningful local structure.



### **2.2. Radius Neighbor Graphs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_B/image_02_02.jpg?v=1783845517" width="250">



>* Connects points within a chosen distance.
>* Handles uneven densities without forced neighbors.

>* Radius and metric shape neighborhood structure.
>* Graphs support clustering, anomalies, and semi-supervision.

>* Graphs support local structure and outlier detection.
>* Standardize features so radius comparisons stay meaningful.



In [ ]:
#@title Python Code - Radius Neighbor Graphs

# Radius graphs connect points within one distance.
# This example uses simple store coordinates.
# The graph reveals dense and sparse areas.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import radius_neighbors_graph

# Set a seed for reproducibility.
np.random.seed(7)

# Create tiny store locations in kilometers.
stores = np.array([
    [0.0, 0.0], [0.8, 0.4], [1.2, -0.3],
    [5.0, 5.0], [5.6, 5.2], [9.0, 1.0]
])

# Choose a radius in kilometers.
radius_km = 1.5

# Build a connectivity graph.
graph = radius_neighbors_graph(
    stores, radius=radius_km, mode="connectivity", include_self=False
)

# Convert sparse graph to a small array.
adjacency = graph.toarray().astype(int)

# Count neighbors for each store.
neighbor_counts = adjacency.sum(axis=1)

# Print a compact summary.
print("Radius:", radius_km, "km")
print("Stores shape:", stores.shape)
print("Neighbor counts:", neighbor_counts.tolist())
print("Isolated store indices:", np.where(neighbor_counts == 0)[0].tolist())

# Plot stores and graph edges.
fig, ax = plt.subplots(figsize=(6, 4))

# Draw edges between connected stores.
rows, cols = np.where(adjacency == 1)
for i, j in zip(rows, cols):
    if i < j:
        ax.plot(
            [stores[i, 0], stores[j, 0]],
            [stores[i, 1], stores[j, 1]],
            color="gray", linewidth=1.5, alpha=0.8
        )

# Draw store points.
ax.scatter(stores[:, 0], stores[:, 1], s=80, color="royalblue")

# Label each store index.
for idx, (x, y) in enumerate(stores):
    ax.text(x + 0.1, y + 0.1, str(idx), fontsize=10)

# Add a clear title and labels.
ax.set_title("Radius Neighbor Graph")
ax.set_xlabel("x position (km)")
ax.set_ylabel("y position (km)")
ax.grid(True, alpha=0.3)

# Keep the plot readable.
plt.tight_layout()
plt.show()



### **2.3. K Neighbor Graphs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_B/image_02_03.jpg?v=1783845512" width="250">



>* Links each point to nearest neighbors
>* Supports clustering, labeling, and similarity analysis

>* Metric and k shape graph structure.
>* Directed or mutual links change meaning.

>* Graphs preserve local structure for efficient analysis.
>* Fixed neighbors create tradeoffs across data densities.



## **3. Neighborhood Components Analysis**

### **3.1. NCA Overview**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_B/image_03_01.jpg?v=1783845526" width="250">



>* NCA learns task-specific feature transformations.
>* It improves nearest-neighbor classification through better distances.

>* Original features can hide class patterns.
>* NCA learns label-guided distances for better neighbors.

>* NCA learns task-specific space for neighbors.
>* Works best with labels and local similarity.



### **3.2. Learning Better Distances**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_B/image_03_02.jpg?v=1783845524" width="250">



>* NCA learns distances better than raw features.
>* It emphasizes class patterns, reducing misleading noise.

>* Reshapes space for better neighbor decisions.
>* Emphasizes useful features, reduces distracting variation.

>* NCA prioritizes class-friendly neighborhoods over visualization.
>* Useful transformations help classification but may overfit.



### **3.3. Learning Better Distances**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_B/image_03_03.jpg?v=1783845527" width="250">



>* NCA learns distances for better neighbors.
>* It reduces noise and highlights useful features.

>* NCA learns which features matter most.
>* It highlights class patterns, reducing distractions.

>* NCA learns class-friendly feature representations.
>* Avoid overfitting; preserve meaningful patterns across data.



# <font color="#418FDE" size="6.5" uppercase>**Advanced Neighbors**</font>


In this lecture, you learned to:
- Use centroid and graph-based neighbor tools for classification and representation tasks. 
- Construct k-neighbor and radius-neighbor graphs for downstream algorithms. 
- Apply Neighborhood Components Analysis to learn transformations that improve neighbor classification. 

In the next Module (Module 14), we will go over 'None'